# Besoin Client 3 – Classification du type d'implantation

Objectif : prédire la colonne `implantation_station` à partir des caractéristiques d'une borne.

Responsable : **Personne 1**

## 1. Imports et chargement

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, f1_score)

df = pd.read_csv('IRVE_cleaned.csv', low_memory=False)
print(df.shape)
df.head()

## 2. Préparation des données

**Justification des colonnes choisies :** (à remplir – analyser la corrélation avec la cible)

In [ ]:
TARGET = 'implantation_station'

# Colonnes features – à adapter
FEATURES = ['puissance_nominale', 'nbre_pdc', 'consolidated_latitude', 'consolidated_longitude']
# + colonnes catégorielles si besoin : 'type_prise', 'region'...

df_model = df[FEATURES + [TARGET]].dropna()
print(f'Shape : {df_model.shape}')
print(df_model[TARGET].value_counts())

In [ ]:
# Encodage de la cible
le = LabelEncoder()
y = le.fit_transform(df_model[TARGET])
X = df_model[FEATURES].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train : {X_train.shape} | Test : {X_test.shape}')

## 3. Entraînement + GridSearchCV

**Justification du choix du modèle :** (à remplir – ex. Random Forest : robuste aux outliers, pas besoin de normalisation, interprétable via feature importance)

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(random_state=42))
])

param_grid = {
    'clf__n_estimators': [100, 200],
    'clf__max_depth':    [None, 10, 20],
    'clf__min_samples_split': [2, 5]
}

grid = GridSearchCV(pipeline, param_grid, cv=5, scoring='f1_weighted', n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)

print(f'Meilleurs paramètres : {grid.best_params_}')
print(f'Meilleur F1 (CV)     : {grid.best_score_:.4f}')

## 4. Métriques d'évaluation

In [ ]:
best = grid.best_estimator_
y_pred = best.predict(X_test)

print('=== Rapport de classification ===')
print(classification_report(y_test, y_pred, target_names=le.classes_))

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Matrice de confusion'); plt.xlabel('Prédit'); plt.ylabel('Réel')
plt.tight_layout(); plt.show()

## 5. Sauvegarde des modèles

In [ ]:
joblib.dump(best, 'model_classification.pkl')
joblib.dump(le,   'label_encoder_classif.pkl')
print('Modèles sauvegardés : model_classification.pkl, label_encoder_classif.pkl')

## 6. Conclusion

- Variables choisies + justification : **à rédiger**
- Modèle + principe de fonctionnement : **à rédiger**
- Analyse des métriques : **à rédiger**